In [1]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    dbname="ProjetoMC536",
    user="postgres",
    password="GSW30_curry",
    host="localhost",
    port="5432"
)

cursor = conn.cursor()

In [11]:
uf_amazonia_legal = ['AC', 'AP', 'AM', 'MA', 'MT', 'PA', 'RO', 'RR', 'TO']

cursor.execute("""
            SELECT sigla, nome FROM public."CategoriaUnidadeConservacao"
        """)
rows = cursor.fetchall()
df_categoria_uc = pd.DataFrame(rows, columns=['sigla', 'nome'])
print(df_categoria_uc)

    sigla                                      nome
0    ESEC                         Estação Ecológica
1   PARNA                           Parque Nacional
2   REBIO                         Reserva Biológica
3     RVS                 Refúgio de Vida Silvestre
4     APA                Área de Proteção Ambiental
5    ARIE     Área de Relevante Interesse Ecológico
6   FLONA                         Floresta Nacional
7     RDS    Reserva de Desenvolvimento Sustentável
8   RESEX                      Reserva Extrativista
9    RPPN  Reserva Particular do Patrimônio Natural
10   MONA                         Monumento Natural


In [24]:
df = pd.read_csv('../datasets/UnidadesConservacao.csv', sep=';')

df = df.rename(columns={
    'Nome da UC': 'nome',
    'Informações Gerais': 'id_estado',
    'Esfera Administrativa': 'id_categoria_unidade_conservacao',
})
df = df[['nome', 'UF', 'id_estado', 'id_categoria_unidade_conservacao']]

for index, row in df.iterrows():
    nome = row['nome']
    uf = row['UF']

    if uf not in uf_amazonia_legal:
        df.drop(index, inplace=True)
        continue
    else:
        cursor.execute("""
            SELECT id_estado FROM public."Estado" WHERE uf = %s
        """, (uf,))
        estado_id = cursor.fetchone()[0]

    # Verifica se a unidade de conservação já existe na tabela
    cursor.execute("""
        SELECT id_categoria_unidade_conservacao
        FROM public."CategoriaUnidadeConservacao"
        WHERE %s ILIKE '%%' || nome || '%%'
    """, (nome,))
    categoria_uc = cursor.fetchall()

    if not categoria_uc:
        cursor.execute("""
        SELECT id_categoria_unidade_conservacao
        FROM public."CategoriaUnidadeConservacao"
        WHERE %s ILIKE '%%' || sigla || '%%'
        """, (nome,))
        categoria_uc = cursor.fetchall()
   
    if categoria_uc:
        categoria_uc = categoria_uc[0][0]
    else:
        categoria_uc = None

    df.at[index, 'id_categoria_unidade_conservacao'] = categoria_uc
    df.at[index, 'id_estado'] = estado_id

df.drop(columns=['UF'], inplace=True)
data = list(df.itertuples(index=False, name=None))
print(data)


[('RESERVA PARTICULAR DO PATRIMÔNIO NATURAL FAZENDA ESTÂNCIA DOROCHÊ', 5.0, 10), ('RESERVA PARTICULAR DO PATRIMÔNIO NATURAL JUBRAN', 5.0, 10), ('RESERVA PARTICULAR DO PATRIMÔNIO NATURAL ESTÂNCIA ECOLÓGICA SESC - PANTANAL', 5.0, 10), ('RESERVA PARTICULAR DO PATRIMÔNIO NATURAL ESTÂNCIA ECOLÓGICA SESC - PANTANAL', 5.0, 10), ('ESTAÇÃO ECOLÓGICA DE TAIAMÃ', 5.0, 1), ('PARQUE ESTADUAL DO GUIRÁ', 5.0, None), ('PARQUE ESTADUAL ENCONTRO DAS ÁGUAS', 5.0, None), ('PARQUE ESTADUAL MARINHO DO PARCEL DE MANUEL LUÍS', 4.0, None), ('PARQUE ESTADUAL MARINHO BANCO DO TAROL', 4.0, None), ('PARQUE ESTADUAL MARINHO BANCO DO ÁLVARO', 4.0, None), ('RESERVA BIOLÓGICA DO LAGO PIRATUBA', 2.0, 3), ('ÁREA DE PROTEÇÃO AMBIENTAL DE ALGODOAL-MAIANDEUA', 6.0, 5), ('REFÚGIO DE VIDA SILVESTRE DA ILHA DO CANELA', 6.0, 4), ('ÁREA DE RELEVANTE INTERESSE ECOLÓGICO PROJETO DINÂMICA BIOLÓGICA DE FRAGMENTOS FLORESTAIS', 3.0, 6), ('PARQUE NACIONAL DE ANAVILHANAS', 3.0, 2), ('ESTAÇÃO ECOLÓGICA DE JUTAÍ-SOLIMÕES', 3.0, 1), ('EST

In [29]:
from psycopg2.extras import execute_values

query = """
    INSERT INTO public."UnidadeConservacao" 
    (nome, id_estado, id_categoria_unidade_conservacao)
    VALUES %s
    ON CONFLICT (nome) DO NOTHING
"""
execute_values(cursor, query, data)
# Finaliza
conn.commit()
cursor.close()
conn.close()

In [28]:
#Caso comando dê erro, desfaz as alterações
conn.rollback()